# Build a controlled noisy gather

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "src/data.py").exists():
    raise RuntimeError("Launch Jupyter from the repository or notebooks folder.")
sys.path.insert(0, str(root))
from src.data import ROOT, LEVELS, load, mix

Use the prepared subset to reproduce a mixture. To reconstruct every aperture from HDF5, run python -m src.data --out runs/prepared. Event IDs are assigned before channel selection.

In [2]:
events, pairs = load(ROOT / "data/prepared")
assert events.event_id.is_unique
display(pd.crosstab(events["array"], events["split"]))

split,pilot,test,train,validation
array,,,,
longvalley,3,133,12,3
ridgecrest,3,12,12,3


In [3]:
i = events.index[events.split == "train"][0]
s, n = pairs["reference"][i], pairs["noise"][i]
checks = []
for level in LEVELS:
    x = mix(s, n, level)
    achieved = 10 * np.log10(np.sum(s**2) / np.sum((x - s) ** 2))
    checks.append((level, achieved))
display(pd.DataFrame(checks, columns=["requested_db", "measured_db"]))

,requested_db,measured_db
0,-5,-5.000000e+00
1,0,-1.446491e-15
2,5,5.000000e+00
3,10,1.000000e+01


The ratio is defined over the entire aperture. Channelwise gains will vary with coupling and noise amplitude.